In [1]:
import gym
#from gym.wrappers import Monitor
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import base64, io

import numpy as np
from collections import deque, namedtuple

# For visualization
from gym.wrappers.monitoring import video_recorder
from IPython.display import HTML
from IPython import display 
import glob

### 환경만들기 

`-` 환경을 만드는 방법은 아래와 같다. 

In [2]:
env = gym.make('LunarLander-v2',new_step_api=True)

- `new_step_api=True`를 쓰지 않으면 귀찮은 warning이 생기니까 시키는대로 해보자.. (무슨 역할인지는 정확하게 모르겠음)

`-` 환경에 대한 기본 정보를 조사하여 보자. 

In [3]:
env.observation_space

Box([-1.5       -1.5       -5.        -5.        -3.1415927 -5.
 -0.        -0.       ], [1.5       1.5       5.        5.        3.1415927 5.        1.
 1.       ], (8,), float32)

- 환경에 대한 상태는 모두 8개의 변수로 표현이 가능하다. 
- 각 변수의 범위는 (-1.5, 1.5), (-1.5, 1.5), (-5, 5), (-5, 5), (-3.14, 3.14), (-5, 5), (0, 1), (0, 1) 이다. 

In [4]:
env.action_space

Discrete(4)

- 이 환경에서 유효한 action은 4개이다. (=에이전트는 4개의 action을 할 수 있다.)

### 환경관찰 

`-` 환경관찰 

In [5]:
env.reset()

array([-0.00238914,  1.4133253 , -0.24200949,  0.10689148,  0.00277521,
        0.05481878,  0.        ,  0.        ], dtype=float32)

In [6]:
env.step(0) # state, reward, done, _ 

(array([-0.00477858,  1.4151527 , -0.24168205,  0.0812076 ,  0.00548406,
         0.05418264,  0.        ,  0.        ], dtype=float32),
 0.5061723043466486,
 False,
 False,
 {})

- 보상이 왜 정의되고 있는것이지? 

### Replay Buffer

`-` Replay buffer를 만들기 위한 예비학습

(예비학습1)

In [7]:
import collections

In [8]:
_memory=collections.deque(maxlen=3)
_memory

deque([])

In [9]:
_memory.append([1,2,3])
_memory.append([11,22,33])
_memory.append([111,222,333])

In [10]:
_memory

deque([[1, 2, 3], [11, 22, 33], [111, 222, 333]])

In [11]:
_memory.append([1111,2222,3333])

In [12]:
_memory

deque([[11, 22, 33], [111, 222, 333], [1111, 2222, 3333]])

(예비학습2)

In [13]:
import random

In [14]:
random.sample(_memory,2)

[[111, 222, 333], [11, 22, 33]]

예비학습끝

---

`-` 랜덤액션을 연속적으로 생성하고 그 결과를 기록해보자. 

In [15]:
states = []
actions = []
rewards = []
next_states = []
dones =[] 

In [16]:
for epsd in range(1, 10): 
    state1 = env.reset() # 환경리셋 + 초기화된 환경을 state라는 변수에 저장 
    ## 게임 1회, 2회.. 
    for frame in range(1500): 
        action = env.action_space.sample() # 매회 랜덤액션을 뽑음 
        state2, reward, done, _, _ = env.step(action) # 액션을 환경에 전달 -> (next_state, reward, done) 을 받음 
    
        ## save the history 
        states.append(state1.tolist())
        actions.append(action)
        rewards.append(reward)
        next_states.append(state2.tolist())
        dones.append(done)

        ## update state 
        state1 = state2 
        if done:
            break 
    ## 
    if epsd % 100 == 0: 
        print("에피소드: {}회".format(epsd))

`-` 모인 히스토리를 확인해보자. 

In [17]:
len(states), len(actions), len(next_states), len(rewards)

(793, 793, 793, 793)

`-` states, actions, next_states, rewards를 모아서 튜플을 만든다. 

### Qnetwork 설계

`-` 네트워크의 목적: 내가 여기서 뭘 해야하는지 알려줘! = 내가 이 상태에서, 어떠한 액션을 해야하는지 알려줘 $\to$ 8개의 상태를 입력으로 받으면 4개의 액션에 대한 좋은 정도를 숫자로 표현하는 어떠한 함수를 만들자. 

`-` net 설계 

In [18]:
net = torch.nn.Sequential(
    torch.nn.Linear(in_features=8, out_features=128),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=128, out_features=64),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=64, out_features=32),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=32, out_features=4)
)
net

Sequential(
  (0): Linear(in_features=8, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=4, bias=True)
)

In [19]:
net(torch.tensor(states))

tensor([[-0.0276,  0.0356, -0.1132,  0.0363],
        [-0.0275,  0.0345, -0.1119,  0.0352],
        [-0.0271,  0.0349, -0.1124,  0.0348],
        ...,
        [-0.0279,  0.0319, -0.0971,  0.0519],
        [-0.0223,  0.0335, -0.1045,  0.0506],
        [-0.0211,  0.0354, -0.1086,  0.0504]], grad_fn=<AddmmBackward0>)

### Policy 설계

`-` 네트워크의 의미

In [20]:
states[0],states[1]

([-0.0063529969193041325,
  1.420130729675293,
  -0.6435121297836304,
  0.4093368351459503,
  0.0073683783411979675,
  0.14576499164104462,
  0.0,
  0.0],
 [-0.012633228674530983,
  1.4287595748901367,
  -0.6334436535835266,
  0.38347136974334717,
  0.012727193534374237,
  0.10718651115894318,
  0.0,
  0.0])

In [21]:
net(torch.tensor(states[0:2]))

tensor([[-0.0276,  0.0356, -0.1132,  0.0363],
        [-0.0275,  0.0345, -0.1119,  0.0352]], grad_fn=<AddmmBackward0>)

- 상태0에서는 액션0이, 상태1에서도 액션0이 가장 좋다는 의미 (왜? q-value가 젤 높으니까..)

`-` 따라서 Agent는 아래와 같이 행동해야 한다. (네트워크가 잘 학습되었다는 전제가 필요함) 
- state[0] -> action = 0 
- state[1] -> action = 0 

In [22]:
net(torch.tensor(states[0:2])).max(axis=1)

torch.return_types.max(
values=tensor([0.0363, 0.0352], grad_fn=<MaxBackward0>),
indices=tensor([3, 3]))

In [23]:
net(torch.tensor(states[0:2])).max(axis=1)[1]

tensor([3, 3])

`-` 네트워크가 있으므로 이제 어떠한 state에 대해서도 뭘 해야할지 (=어떤 액션을 해야할지) 알 수 있다. 

In [24]:
state1 # 어떤 state에 대해서도.. 

array([-0.757498  , -0.20118985, -1.623102  , -0.29303217,  1.261518  ,
        3.1040583 ,  0.        ,  1.        ], dtype=float32)

In [25]:
net(torch.tensor(state1)) # q-value를 계산할 수 있고 

tensor([-0.0610,  0.0152, -0.1486,  0.0499], grad_fn=<AddBackward0>)

In [26]:
int(torch.argmax(net(torch.tensor(state1)))) # 그래서 다음에 우리가 어떤행동을 해야할 지 알 수 있음

3

### 학습

`-` 네트워크를 학습시키자. 

In [28]:
gamescore=[]
eps = 0.05
opt = torch.optim.Adam(net.parameters(),lr=0.0001)

In [29]:
for epsd in range(1000): # 게임 2000판 시켜줌.. 
    gamescore.append(0)
    state1 = env.reset() # 환경리셋 + 초기화된 환경을 state라는 변수에 저장 
    for frame in range(1500): # 게임1판당 max 1500프레임만 할 수 있음 (그러니까 최대 30초) 
        
        # (step1) Agent: action 
        if np.random.rand() < eps: 
            action = env.action_space.sample() # 매회 랜덤액션을 뽑음 --> 폐기 
        else:
            action = int(torch.argmax(net(torch.tensor(state1)))) # 네트워크가 알려주는 action을 뽑음 
        
        # (step2) Agent -> Env // Env -> Agent 
        state2, reward, done, _, _ = env.step(action) # 액션을 환경에 전달 -> (next_state, reward, done) 을 받음 
        reward = reward #+ frame/1000
        
        # (step3) Agnet: save data and learn 
        ## save data 
        states.append(state1)
        actions.append(action)
        rewards.append(reward)
        next_states.append(state2)
        ## prepare recent data 
        _states = torch.tensor(random.sample(states[-1000000:],128))
        _actions = torch.tensor(random.sample(actions[-100000:],128)).reshape(-1,1)
        _next_states = torch.tensor(random.sample(next_states[-100000:],128))
        _rewards = torch.tensor(random.sample(rewards[-100000:],128)).reshape(-1,1)
        _dones = np.array(random.sample(dones[-100000:],128)).reshape(-1,1) 
        ## leanrn with pytorch 
        yhat = net(_states).gather(1,_actions) ## (s,a) -> q(s,a) // 내가 현재상태 state에서, 현재 action을 하여 얻을 것이라 예상하는 보상 (net가 알려주는) 
        y = _rewards + 0.99 * net(_next_states).detach().max(1)[0].unsqueeze(1)*(1-_dones) ## 그런데 실제로는 이게 맞다고 봐야지~ 
        loss = torch.mean((y-yhat)**2)
        #loss = loss_fn(y,yhat)
        loss.backward()
        
        opt.step()
        opt.zero_grad()

        # (step4) Agent: prepare next steps 
        state = state2  
        #eps = max(0.01, 0.995*eps) 
        gamescore[epsd] = gamescore[epsd]+reward
        
        # terminate 
        if done:
            break 
    print('\rEpisode {}\tAverage Score: {:.2f}'.format(epsd, np.mean(gamescore[-100:])), end="")
    if epsd % 100 == 0:
        print('\rEpisode {}\tAverage Score: {:.2f}'.format(epsd, np.mean(gamescore[-100:])))

/home/cgb4/anaconda3/envs/py37/lib/python3.7/site-packages/ipykernel_launcher.py:23: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at  /opt/conda/conda-bld/pytorch_1656352464346/work/torch/csrc/utils/tensor_new.cpp:201.)


Episode 0	Average Score: -374.57
Episode 100	Average Score: -481.70
Episode 200	Average Score: -487.50
Episode 300	Average Score: -485.57
Episode 400	Average Score: -487.42
Episode 500	Average Score: -469.92
Episode 600	Average Score: -553.87
Episode 700	Average Score: -412.56
Episode 800	Average Score: -460.41
Episode 900	Average Score: -502.11
Episode 999	Average Score: -618.61